# HS4002 Week 6

## OLS with Higher-Order Effects, Interaction Terms, and Logistic Regression

Today we cover:
1. Polynomial terms in OLS (age-squared)
2. Interaction effects
3. The Linear Probability Model (LPM)
4. Logistic regression
5. Marginal effects


In [24]:
%pip install pandas numpy plotnine statsmodels marginaleffects stargazer

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
raw_df = pd.read_csv("gss2022_mini.csv")

chosen_variables = [
	'id', 'age', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat',
	'income16', 'prestg10', 'degree', 'race', 'sex', 'wrkstat'
]
df = raw_df[chosen_variables].dropna().copy()

# Recode income16 to continuous midpoint values
income_map = {
	1: 500,    2: 2000,   3: 3500,   4: 4500,   5: 5500,   6: 6500,
	7: 7500,   8: 9000,   9: 11250,  10: 13750, 11: 16250, 12: 18750,
	13: 21250, 14: 23750, 15: 27500, 16: 32500, 17: 37500, 18: 45000,
	19: 55000, 20: 67500, 21: 82500, 22: 100000, 23: 120000, 24: 140000,
	25: 160000, 26: 250000
}
df['income_cont'] = df['income16'].map(income_map)

# Binary cultural participation indicators
df['binary_lvmus']  = (df['yrlvmus']  == 1).astype(int)
df['binary_artxbt'] = (df['yrartxbt'] == 1).astype(int)
df['binary_movie']  = (df['yrmovie']  == 1).astype(int)
df['binary_creat']  = (df['yrcreat']  == 1).astype(int)
df['omni'] = df[['binary_lvmus','binary_artxbt','binary_movie','binary_creat']].sum(axis=1)

# Derived variables
df['race_bin']   = df['race'].astype(str)
df['ba_binary']  = (df['degree'] >= 3).astype(int)
df['sex_woman']  = np.where(df['sex'] == 2, 'Woman', 'Not Woman')
df['working']    = np.where(df['wrkstat'] <= 3, 'Working', 'Not Working')

# OLS with Higher-Order Polynomials

Begin by regressing income on age. Save as `ols_model1`.

In [26]:
ols_model1 = smf.ols('income_cont ~ age', data=df).fit()
ols_model1.summary()
# interpretation: We observe no associations between age and income (p = 0.892) , ceteris paribus

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            income_cont   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                   0.01847
Date:                Wed, 11 Nov 2026   Prob (F-statistic):              0.892
Time:                        23:10:38   Log-Likelihood:                -8910.5
No. Observations:                 707   AIC:                         1.782e+04
Df Residuals:                     705   BIC:                         1.783e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   8.632e+04   8546.726     10.099      0.000    6.95e+04    1.03e+05
age           21.2431    156.330      0.136      0.892    -285.684     328.171
==============================================================================
Omnibus:                      113.766   Durbin-Watson:                   1.629
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              168.872
Skew:                           1.178   Prob(JB):                     2.14e-37
Kurtosis:                       3.430   Cond. No.                         172.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Adding an age-squared term

In Python formulas, use `I(age**2)` — the `I()` wrapper tells the formula parser to treat the expression literally.

In [27]:
# this is assuming it is not a linear model! the age^2 allows line to bend
# adding age^2 asks if income goes up with age, then comes down
ols_model2 = smf.ols('income_cont ~ age + I(age**2)', data=df).fit()
ols_model2.summary()
# income = aage^2 + bage + c
# income = -37.5age^2 + 3886age - 2088
# peak age occurs when rate of change is 0. dy/dx = 0
# dy/dx = 2(-37.5)age + 3886 = 0
# peak_age = -3886/2(-37.5) = 52 (round up)
# Income rises with age until about 52, then gently falls. The pattern is statistically real, but age alone explains only about 2% of why people's incomes differ.

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            income_cont   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     7.568
Date:                Wed, 11 Nov 2026   Prob (F-statistic):           0.000560
Time:                        23:10:38   Log-Likelihood:                -8903.0
No. Observations:                 707   AIC:                         1.781e+04
Df Residuals:                     704   BIC:                         1.783e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept   -2088.6030   2.43e+04     -0.086      0.931   -4.97e+04    4.55e+04
age          3886.3291   1006.049      3.863      0.000    1911.113    5861.545
I(age ** 2)   -37.4677      9.636     -3.888      0.000     -56.387     -18.548
==============================================================================
Omnibus:                      109.972   Durbin-Watson:                   1.611
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              160.918
Skew:                           1.148   Prob(JB):                     1.14e-35
Kurtosis:                       3.440   Cond. No.                     3.16e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.16e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

## Making a regression table

As in Week 5, the `stargazer` package collects fitted models into a single regression table. Reading models side by side is much easier than scrolling through two separate `.summary()` blocks.

In [28]:
from stargazer.stargazer import Stargazer

sg = Stargazer([ols_model1, ols_model2])
sg.title('Income, Age, and Age-Squared')

# Label the models and variables readably, rather than by raw column name
sg.dependent_variable_name('Income (USD)')

sg.custom_columns(['Model 1', 'Model 2'], [1, 1])
sg.covariate_order(['age', 'I(age ** 2)', 'Intercept'])
sg.rename_covariates({
	'age':         'Age',
	'I(age ** 2)': 'Age-squared',
	'Intercept':   'Intercept'
})

# Ending the cell on the object itself renders the table in the notebook
sg

In [29]:
person_a = 15
person_b = 45
(person_b - person_a)*3886 + ((person_b - person_a)**2)*(-37.5)
# 3886 is the main effect, 37.5 is the quadratic effect
# Person B is expected to have $82830 more than Person B

82830.0

## Interaction Effects

Add BA education and gender to the model. Save as `ols_model3`.

In [30]:
ols_model3 = smf.ols('income_cont ~ age + I(age**2) + ba_binary + sex_woman', data=df).fit()
ols_model3.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            income_cont   R-squared:                       0.203
Model:                            OLS   Adj. R-squared:                  0.199
Method:                 Least Squares   F-statistic:                     44.79
Date:                Wed, 11 Nov 2026   Prob (F-statistic):           1.61e-33
Time:                        23:10:38   Log-Likelihood:                -8830.1
No. Observations:                 707   AIC:                         1.767e+04
Df Residuals:                     702   BIC:                         1.769e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept          -8893.2601   2.21e+04     -0.402      0.688   -5.23e+04    3.46e+04
sex_woman[T.Woman] -1.663e+04   4871.657     -3.414      0.001   -2.62e+04   -7067.441
age                 3275.5520    910.156      3.599      0.000    1488.599    5062.505
I(age ** 2)          -31.3456      8.720     -3.595      0.000     -48.466     -14.225
ba_binary            6.04e+04   4874.146     12.393      0.000    5.08e+04       7e+04
==============================================================================
Omnibus:                      102.337   Durbin-Watson:                   1.765
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              145.602
Skew:                           1.046   Prob(JB):                     2.42e-32
Kurtosis:                       3.751   Cond. No.                     3.19e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.19e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Now add an interaction term between BA and gender. In statsmodels formulas, `:` creates an interaction.

Save as `ols_model4`.

In [31]:
# this model asks if education pays off equally for men and women. 
ols_model4 = smf.ols(
	'income_cont ~ age + I(age**2) + ba_binary + sex_woman + ba_binary:sex_woman',
	data=df
).fit()
ols_model4.summary()
# education pays off less for women. for men, education adds $69623 to their income
# but for women, education adds $69623-$17474=$52149 to their income. 
# but p > 0.1, so evidence is weak

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            income_cont   R-squared:                       0.207
Model:                            OLS   Adj. R-squared:                  0.201
Method:                 Least Squares   F-statistic:                     36.59
Date:                Wed, 11 Nov 2026   Prob (F-statistic):           2.37e-33
Time:                        23:10:38   Log-Likelihood:                -8828.5
No. Observations:                 707   AIC:                         1.767e+04
Df Residuals:                     701   BIC:                         1.770e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                    -1.184e+04   2.22e+04     -0.534      0.593   -5.53e+04    3.17e+04
sex_woman[T.Woman]           -8323.3760   6712.340     -1.240      0.215   -2.15e+04    4855.323
age                           3221.5326    909.213      3.543      0.000    1436.425    5006.640
I(age ** 2)                    -30.7979      8.712     -3.535      0.000     -47.902     -13.694
ba_binary                     6.962e+04   7073.827      9.843      0.000    5.57e+04    8.35e+04
ba_binary:sex_woman[T.Woman] -1.748e+04   9733.988     -1.796      0.073   -3.66e+04    1626.933
==============================================================================
Omnibus:                      102.870   Durbin-Watson:                   1.768
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              146.972
Skew:                           1.043   Prob(JB):                     1.22e-32
Kurtosis:                       3.796   Cond. No.                     3.21e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.21e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [32]:
from stargazer.stargazer import Stargazer

sg = Stargazer([ols_model3, ols_model4])
sg.title('Interaction Between BA and Gender')
sg.dependent_variable_name('Income (USD)')

sg.custom_columns(['Model 3', 'Model 4'], [1, 1])
sg.covariate_order([
	'age',
	'I(age ** 2)',
	'ba_binary',
	'sex_woman[T.Woman]',
	'ba_binary:sex_woman[T.Woman]',
	'Intercept'
])
sg.rename_covariates({
	'age':                          'Age',
	'I(age ** 2)':                  'Age-squared',
	'ba_binary':                    'BA degree',
	'sex_woman[T.Woman]':           'Gender (woman = 1)',
	'ba_binary:sex_woman[T.Woman]': 'BA × Woman',
	'Intercept':                    'Intercept'
})

sg

Apart from the main effects of age, education and gender, we also observe an interaction effect between education and gender. For women, each year of education is associated with a decreased income of $17484 when compared to men (p < 0.1).

# Using the Linear Probability Model (LPM)

The LPM uses OLS on a binary dependent variable. It's quick and interpretable, though it can predict probabilities outside [0, 1].

Regress live music attendance (`binary_lvmus`) on BA education and log income.

In [33]:
lpm_model1 = smf.ols(
	'binary_lvmus ~ ba_binary + np.log(income_cont)',
	data=df
).fit()
print(lpm_model1.summary())

# with lpm, binary_lvmus only takes 2 values, 0 or 1
# this model is describing the the relationship between income and the probability of live music attendance
# Having a BA is expected to be associated with a 0.128 (p < 0.01) increase in the probability of attending live music, holding all else constant
# an increase in one unit of income is expected to be asscoiated with a 0.0887 (p < 0.01) increase in the probability of attending live music, holding all else constant

                            OLS Regression Results                            
Dep. Variable:           binary_lvmus   R-squared:                       0.069
Model:                            OLS   Adj. R-squared:                  0.066
Method:                 Least Squares   F-statistic:                     26.08
Date:                Wed, 11 Nov 2026   Prob (F-statistic):           1.19e-11
Time:                        23:10:39   Log-Likelihood:                -486.80
No. Observations:                 707   AIC:                             979.6
Df Residuals:                     704   BIC:                             993.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -0.5077    

# Using a Logit Model

Now, let's try logistic regression.

In [34]:
# logistic regression helps ensure the y-value stays between 0 and 1 by logging it
# this model is describing the relationship between income and the log odds of attendance. thus LPM and logit models are not comparable

# careful: how large the coefficient is does not matter. since model is logged, it is difficult to predict the impact the size of coefficient has on y
# to understand how coefficient impacts y for logit models, look at marginal effects instead!
logit_model1 = smf.logit(
	'binary_lvmus ~ ba_binary + np.log(income_cont)',
	data=df
).fit()
print(logit_model1.summary())

# computer took 5 rounds to guenss the answer
# pseudo R-squared = 0.05151. model improves on a model with no predictors (i.e. LL-Null value) by 5%

Optimization terminated successfully.
         Current function value: 0.655998
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:           binary_lvmus   No. Observations:                  707
Model:                          Logit   Df Residuals:                      704
Method:                           MLE   Df Model:                            2
Date:                Wed, 11 Nov 2026   Pseudo R-squ.:                 0.05151
Time:                        23:10:39   Log-Likelihood:                -463.79
converged:                       True   LL-Null:                       -488.98
Covariance Type:            nonrobust   LLR p-value:                 1.151e-11
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -4.4595      0.961     -4.639      0.000      -6.344      -2.575
ba

a one unit increase in education is expected to cause an increase in the log odds of attending live music by 0.5186 (p < 0.01), holding all else constant. a one unit increase in income is expected to cause an increase in the log odds of attending live music by 0.3938(p < 0.001), holding all else constant.

Build a second logit model adding gender, age, and age-squared. Compare both.

In [35]:
logit_model2 = smf.logit(
	'binary_lvmus ~ ba_binary + np.log(income_cont) + sex_woman + age + I(age**2)',
	data=df
).fit()
print(logit_model2.summary())

Optimization terminated successfully.
         Current function value: 0.645323
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:           binary_lvmus   No. Observations:                  707
Model:                          Logit   Df Residuals:                      701
Method:                           MLE   Df Model:                            5
Date:                Wed, 11 Nov 2026   Pseudo R-squ.:                 0.06695
Time:                        23:10:39   Log-Likelihood:                -456.24
converged:                       True   LL-Null:                       -488.98
Covariance Type:            nonrobust   LLR p-value:                 8.952e-13
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -3.6710      1.156     -3.176      0.001      -5.936      -1.405
se

In [36]:
# compare lpm and logit models using stargazer

from stargazer.stargazer import Stargazer

sg = Stargazer([lpm_model1, logit_model1, logit_model2])
sg.title('Live Music Attendance: LPM vs Logit')
sg.dependent_variable_name('Attends live music')

sg.custom_columns(['LPM', 'Logit 1', 'Logit 2'], [1, 1, 1])
sg.covariate_order([
	'ba_binary',
	'np.log(income_cont)',
	'sex_woman[T.Woman]',
	'age',
	'I(age ** 2)',
	'Intercept'
])
sg.rename_covariates({
	'ba_binary':           'BA degree',
	'np.log(income_cont)': 'log(Income)',
	'sex_woman[T.Woman]':  'Gender (woman = 1)',
	'age':                 'Age',
	'I(age ** 2)':         'Age-squared',
	'Intercept':           'Intercept'
})

# The LPM coefficients are changes in probability, the logit coefficients are
# log-odds, so compare signs and significance across columns — not magnitudes.
sg

# the tables look very different as the models are measuring different things -- one is measuring probability, the rest are measuring log odds

## Comparing Logit Models

### AIC comparison

In [37]:
print(f'AIC logit_model1: {logit_model1.aic:.2f}')
print(f'AIC logit_model2: {logit_model2.aic:.2f}')
# model with smaller AIC is better

AIC logit_model1: 933.58
AIC logit_model2: 924.49


### Likelihood ratio test


In [ ]:
# use this to compare logit nested models

from scipy.stats import chi2

lr_stat = 2 * (logit_model2.llf - logit_model1.llf)
df_diff = logit_model2.df_model - logit_model1.df_model
p_value = chi2.sf(lr_stat, df_diff)
print(f'LR statistic = {lr_stat:.4f}')
print(f'df = {df_diff:.0f}')
print(f'p-value = {p_value:.4f}')

# null hypothesis is that smaller model is the true model. if p is small, reject the null hypothesis 
# since p-value is < 0.01, we reject the null hypothesis. adding the variables gender, age and age-squared made the model fit better

LR statistic = 15.0945
df = 3
p-value = 0.0017


# Marginal Effects

The Python `marginaleffects` package mirrors R's package of the same name.

In [44]:
%pip install marginaleffects
from marginaleffects import avg_comparisons, comparisons


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Average marginal effect

The average marginal effect of a BA degree — equivalent to R's `avg_comparisons(logit_model1, variables='ba_binary')`.

In [ ]:
# prefer using this as it makes more sense than imagining a hypothetical guy (which is what marginal effect at the mean does)
avg_comparisons(logit_model1, variables='ba_binary')
# average marginal effect for evrey bachelor's degree and get the average of that
# contrast: comparing having education (1) with no education (0)
# estimate is the B
# statistic is the z-value

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""ba_binary""","""1 - 0""",0.123828,0.040752,3.03855,0.002465,8.664096,0.043817,0.203839


Interpretation: having a BA degree is associated, on average across the sample, with an increase of 0.1238 in the probability of attending live music, holding all else constant. at p = 0.002365, we reject the null hypothesis that there is no effect at the 0.05 threshold.

## Marginal effect at the mean

The marginal effect for a hypothetical person at the average of all X variables.

In [41]:
comparisons(logit_model1, variables='ba_binary', newdata='mean')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""ba_binary""","""1 - 0""",0.126466,0.041026,3.082541,0.002133,8.873215,0.045917,0.207014


having a BA degree is associated, for a typical average person, with an increase of 0.1264 in the probability of attending live music, holding all else constant. at p = 0.002133, we reject the null hypothesis that there is no effect at the 0.05 threshold

In [42]:
# average marginal effect of income from logit 1 model 
avg_comparisons(logit_model1, variables='income_cont')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_cont""","""+1""",0.000003,4.1291e-7,7.88037,1.2434e-14,46.192645,0.000002,0.000004


every unit of income is associated with, on average across the sample, with an increase of 0.000003 in the probability of attending live music, holding all else constant. at p = 1.2434e, we reject the null hypothesis that there is no effect at the 0.05 threshold

In [43]:
# regressing the impact of sex and income on whether or not you watch a movie

lpm_model5 = smf.ols(
	'binary_movie ~ sex_woman + np.log(income_cont)',
	data=df
).fit()
print(lpm_model5.summary())

                            OLS Regression Results                            
Dep. Variable:           binary_movie   R-squared:                       0.037
Model:                            OLS   Adj. R-squared:                  0.034
Method:                 Least Squares   F-statistic:                     13.57
Date:                Wed, 11 Nov 2026   Prob (F-statistic):           1.65e-06
Time:                        23:10:42   Log-Likelihood:                -499.73
No. Observations:                 707   AIC:                             1005.
Df Residuals:                     704   BIC:                             1019.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -0.4055    

In [46]:
!jupyter nbconvert --to pdf week6_notebook_student.ipynb

[NbConvertApp] Converting notebook week6_notebook_student.ipynb to pdf
[NbConvertApp] Writing 86343 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | bibtex had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 95336 bytes to week6_notebook_student.pdf
